## **OceanOSSE:** Development Notebook

### **Description:**

Notebook to develop & evaluate basic pyinterp Regridders in OceanOSSE.

### **Created By:**

Ollie Tooth (oliver.tooth@noc.ac.uk)

In [1]:
import logging

import numpy as np
import pandas as pd
import pyinterp
import xarray as xr
from tqdm import tqdm
from xarray.indexes import NDPointIndex
from xoak import SklearnGeoBallTreeAdapter

from OceanOSSE.io.dataloader import NetCDFDataLoader

In [ ]:
logger = logging.getLogger(__name__)

logging.basicConfig(
    format="⦿══⦿  OceanOSSE  ⦿══⦿  ║ %(levelname)10s ║ %(asctime)s ║ %(message)s",
    level=logging.INFO,
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.FileHandler("OceanOSSE.log"), logging.StreamHandler()],
)

### Example configuration .toml file read as dict

In [ ]:
config = {
    "domain": {
        "dimensions": {"lev": "deptht", "j": "y", "i": "x"},
        "coordinates": {
            "lat": "nav_lat",
            "lon": "nav_lon",
            "depth": "nav_lev",
        },
        "variables": {
            "tmask": {
                "path": "/dssgfs01/scratch/npd/simulations/Domains/eORCA025/eORCA025_ERA5v1_domain_cfg_mesh_mask_util.nc",
                "open_kwargs": {"engine": "netcdf4"},
            },
        },
    },
    "inputs": {
        "dimensions": {"time": "time_counter", "lev": "deptht", "j": "y", "i": "x"},
        "coordinates": {
            "lat": "nav_lat",
            "lon": "nav_lon",
            "depth": "deptht",
            "time": "time_counter",
        },
        "variables": {
            "thetao_con": {
                "path": "/dssgfs01/scratch/npd/simulations/eORCA1_ERA5_v1/eORCA1_ERA5_1m_grid_T_202*.nc",
                "open_kwargs": {"engine": "netcdf4"},
            },
            "so_abs": {
                "path": "/dssgfs01/scratch/npd/simulations/eORCA1_ERA5_v1/eORCA1_ERA5_1m_grid_T_202*.nc",
                "open_kwargs": {"engine": "netcdf4"},
            },
        },
    },
    "climatology": {
        "read_climatology": False,
        "dimensions": {"month": "time_counter", "lev": "deptht", "j": "y", "i": "x"},
        "coordinates": {
            "lat": "nav_lat",
            "lon": "nav_lon",
            "depth": "deptht",
            "month": "time_counter",
        },
        "variables": {
            "thetao_con": {
                "path": "/dssgfs01/scratch/npd/simulations/eORCA1_ERA5_v1/eORCA1_ERA5_1m_grid_T_202*.nc",
                "open_kwargs": {"engine": "netcdf4"},
            },
            "so_abs": {
                "path": "/dssgfs01/scratch/npd/simulations/eORCA1_ERA5_v1/eORCA1_ERA5_1m_grid_T_202*.nc",
                "open_kwargs": {"engine": "netcdf4"},
            },
        },
    },
    "sampling": {
        "name": "test",
        "error_kernels": [{"name": "test", "kwargs": {"argument": None}}],
    },
    "regridding": {
        "name": "idw",
        "depth_max": 2000.0,
        "mask_name": "tmask",
        },
    "outputs": {
        "output_dir": "/dssgfs01/working/otooth/Software/OceanOSSE/OceanOSSE",
        "output_name": "OceanOSSE_TEST",
        "date_format": "M",
        "chunks": {"time_counter": 12},
        "writer_kwargs": {"unlimited_dims": "time_counter", "mode": "w"},
    },
}

config

### Data Loaders - Input Scalar Fields

In [ ]:
dataloader = NetCDFDataLoader.from_config(config=config, table="inputs")

In [ ]:
ds = dataloader.load_data()
ds = ds.sel(time=slice("2021-01", "2023-12"))
ds

### Data Loaders - Climatologies

In [ ]:
ds_clim = dataloader.compute_monthly_climatology().sel(month=ds['time'].dt.month)
ds_clim

In [ ]:
ds["thetao_con_anom"] = ds["thetao_con"] - ds_clim["thetao_con"]
ds["so_abs_anom"] = ds["so_abs"] - ds_clim["so_abs"]
ds["thetao_con_anom"]

### ObsSampling

In [ ]:
# Load Argo profiles from EN4.2.2 .parquet file:
df_argo = pd.read_parquet("https://noc-msm-o.s3-ext.jc.rl.ac.uk/ocean-obs/OceanOSSE/EN.4.2.2.f.profiles.g10.2001_2026.parquet")

# df_argo = df_argo[(df_argo["JULD"] > pd.Timestamp("2021-01-01")) & (df_argo["JULD"] < pd.Timestamp("2024-01-01"))]

df_argo

In [ ]:
# Load Argo profiles from EN4.2.2 .parquet file:
df_argo = pd.read_parquet("https://noc-msm-o.s3-ext.jc.rl.ac.uk/ocean-obs/OceanOSSE/EN.4.2.2.f.profiles.g10.2001_2026.parquet")
ds_argo = xr.Dataset.from_dataframe(df_argo).rename({"JULD": "time", "LATITUDE": "lat", "LONGITUDE": "lon", "ID": "id"})

ds_argo.time.dt.month

In [ ]:
# Emulate basic ObsSampler (NNSampler) to produce xr.Dataset of Argo profiles:
ds = (ds
      .assign_coords({"lon": ds["lon"],
                      "lat": ds["lat"],
                      "i": ds["i"],
                      "j": ds["j"],
                      "lev": ds["lev"],
                     })
      .set_xindex(("lat", "lon"), 
                  NDPointIndex, 
                  tree_adapter_cls=SklearnGeoBallTreeAdapter
                  )
     )

ds_prof = ds.sel(time=xr.DataArray(df_argo['JULD'].to_numpy().astype('datetime64[ns]'), dims="profile"),
                 lon=xr.DataArray(df_argo["LONGITUDE"].to_numpy(), dims="profile"),
                 lat=xr.DataArray(df_argo["LATITUDE"].to_numpy(), dims="profile"),
                 method="nearest"
                 )

ds_prof = ds_prof.assign_coords(t=xr.DataArray(data=np.searchsorted(ds['time'].values, ds_prof['time'].values),
                                               dims="profile"
                                               ))

ds_prof

### pyinterp IDW Regridder

In [ ]:
from OceanOSSE.gridding.regridder import IDWRegridder

regridder = IDWRegridder.from_config(config=config)

regridder

In [ ]:
ds["tmask"] = ds["thetao_con"].isel(time=0).notnull()
ds

In [ ]:
regridder.regrid(ds_obs=ds_prof, ds_mdl=ds)

In [ ]:
# Load profile and target model grid data:
prof_lons = ds_prof["lon"].values
prof_lats = ds_prof["lat"].values
prof_times = ds_prof["time"].values
prof_values = ds_prof["thetao_con_anom"].values

grid_mask = ds["thetao_con"].isel(time=0).notnull().values
grid_lons = ds["lon"].values.flatten()
grid_lats = ds["lat"].values.flatten()
grid_coords = np.vstack((grid_lons, grid_lats)).T
grid_shape = ds["lon"].shape

In [ ]:
# Iterate over unique profile times and perform IDW interpolation for each depth level:
times = np.unique(prof_times)
da = xr.full_like(ds["thetao_con"], fill_value=np.nan)

pbar = tqdm(
    range(len(times)),
    desc="Interpolation Progress",
    unit="time-step",
)

for n in pbar:
    pbar.set_postfix(time=str(times[n])[:10])
    time_mask = prof_times == times[n]
    lons = prof_lons[time_mask]
    lats = prof_lats[time_mask]
    coords = np.vstack((lons, lats)).T

    for k in range(54):
        data = prof_values[time_mask, k]
        nan_mask = ~np.isnan(data)

        mesh = pyinterp.RTree3D()
        mesh.packing(coords[nan_mask], data[nan_mask])

        idw, _ = pyinterp.inverse_distance_weighting(
            mesh,
            grid_coords,
            boundary_check="none",
            k=30,
            radius=5,
            num_threads=0,
        )

        da.data[n, k, :, :] = idw.reshape(grid_shape)


da

In [ ]:
# Reconstruct the field by adding the monthly climatology back to the regridded anomalies:
da_regrid = da.fillna(0).where(grid_mask) + ds_clim["thetao_con"]

da_regrid[0, 0, :, :].plot()